# HAM10000 — Stage 2: Preprocessing + Split

**Goal:** Stratified train/val/test split, image transforms with justified augmentation,
DataLoader setup, and class-weight computation for weighted CrossEntropyLoss.

**Split:** 70 / 15 / 15 — chosen to keep enough minority-class images in val/test
for stable recall metrics (the rarest class `df` has only 115 images total).

## Cell 1 — Clone / update repo and set up paths

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/Dev252001/HAM10000.git"
REPO_DIR = "/content/ham10000-classifier"
SRC_DIR  = os.path.join(REPO_DIR, "src")
DATA_DIR = os.path.join(REPO_DIR, "data")

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print("Repo cloned.")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    print("Repo updated.")

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print(f"src/ on path: {SRC_DIR}")

## Cell 2 — Install dependencies

In [ ]:
!pip install -q \
  "numpy>=2.0" \
  "pandas>=2.2.2" \
  "Pillow>=10.4.0" \
  "scikit-learn>=1.5.0" \
  "matplotlib>=3.9.0" \
  "seaborn>=0.13.2" \
  "kaggle>=1.6.14" \
  "ipywidgets>=8.1.3"
print("Dependencies ready.")

## Cell 3 — Dataset (Google Drive cache + Kaggle fallback)

**First run:** uploads `kaggle.json`, downloads the dataset (~10 min), saves to Google Drive.  
**All future runs:** copies instantly from Drive — no re-upload, no re-download.

In [ ]:
import os, json, shutil
from google.colab import drive
from data_loader import download_dataset

DRIVE_DATA = "/content/drive/MyDrive/HAM10000_data"
DATA_DIR   = "/content/ham10000-classifier/data"

drive.mount('/content/drive')

if os.path.exists(os.path.join(DRIVE_DATA, 'HAM10000_metadata.csv')):
    print('Dataset found on Drive — copying to /content/ ...')
    if os.path.exists(DATA_DIR):
        shutil.rmtree(DATA_DIR)
    shutil.copytree(DRIVE_DATA, DATA_DIR)
    print('Done ✓')
else:
    print('Dataset not on Drive — downloading from Kaggle (one-time, ~10 min)...')
    from google.colab import files as colab_files
    uploaded = colab_files.upload()
    uploaded_name = list(uploaded.keys())[0]
    creds = json.loads(uploaded[uploaded_name])
    os.makedirs('/root/.kaggle', exist_ok=True)
    with open('/root/.kaggle/kaggle.json', 'w') as f:
        json.dump(creds, f)
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    print(f'Kaggle credentials configured (user: {creds["username"]}).')
    download_dataset(DATA_DIR)
    print('Saving to Google Drive for future sessions...')
    shutil.copytree(DATA_DIR, DRIVE_DATA)
    print('Saved to Drive ✓ — future sessions will skip the Kaggle download')

from data_loader import load_metadata, CLASSES, LABEL_MAP

df = load_metadata(DATA_DIR)
print(f"Total rows: {len(df)} | Classes: {df['dx'].nunique()} | Missing paths: {df['filepath'].isna().sum()}")
df.head(3)

## Cell 4 — Stratified train / val / test split

We use **70 / 15 / 15** split stratified on `dx`.

**Why 70/15/15 instead of 80/10/10?**  
The rarest class (`df`) has only 115 images. With 80/10/10 that gives ~11 images per eval set — too few for stable recall estimates. 70/15/15 gives ~17 per eval set, which is more reliable.

In [ ]:
import pandas as pd
from preprocessing import make_splits

train_df, val_df, test_df = make_splits(df, val_size=0.15, test_size=0.15)

print(f"Train: {len(train_df):>5} images ({len(train_df)/len(df)*100:.1f}%)")
print(f"Val:   {len(val_df):>5} images ({len(val_df)/len(df)*100:.1f}%)")
print(f"Test:  {len(test_df):>5} images ({len(test_df)/len(df)*100:.1f}%)")

## Cell 5 — Verify stratification: class % should be ~equal across all splits

Each split should reflect roughly the same class percentages as the full dataset.
If stratification worked, every row in the table should be close to the `Full` column.

In [ ]:
import pandas as pd

def class_dist(df, name):
    counts = df['dx'].value_counts()
    pct    = (counts / len(df) * 100).round(2)
    return pd.DataFrame({f"{name} n": counts, f"{name} %": pct})

dist = pd.concat([
    class_dist(df,       "Full"),
    class_dist(train_df, "Train"),
    class_dist(val_df,   "Val"),
    class_dist(test_df,  "Test"),
], axis=1).reindex(CLASSES)

dist.index = [LABEL_MAP[c] for c in dist.index]
print("Class distribution across splits (% should be ~equal across Full / Train / Val / Test):")
print(dist.to_string())

## Cell 6 — Compute class weights

Formula: `w_c = N / (C × n_c)`  
Rare classes (like `df`, `vasc`) get higher weights so the loss treats all classes equally.
This tensor will be passed directly to `nn.CrossEntropyLoss(weight=class_weights)` in Stage 3.

In [ ]:
from preprocessing import compute_class_weights
from data_loader import CLASSES, LABEL_MAP

class_weights = compute_class_weights(train_df)

print("Class weights (ordered by CLASS_TO_IDX — alphabetical):")
print(f"{'Class':<8} {'Full Name':<30} {'Train n':>8} {'Weight':>8}")
print("-" * 60)
counts = train_df['dx'].value_counts()
for i, cls in enumerate(CLASSES):
    print(f"{cls:<8} {LABEL_MAP[cls]:<30} {counts.get(cls,0):>8} {class_weights[i].item():>8.4f}")

print(f"\nTensor shape: {class_weights.shape}")
print(f"Tensor dtype: {class_weights.dtype}")
print(f"\nHighest weight → rarest class (expected: df or vasc)")
max_idx = class_weights.argmax().item()
print(f"  → {CLASSES[max_idx]} ({LABEL_MAP[CLASSES[max_idx]]}) with weight {class_weights[max_idx]:.4f}")

## Cell 7 — Inspect transforms pipeline

In [ ]:
from preprocessing import get_transforms

print("=== TRAIN transforms ===")
print(get_transforms('train'))
print()
print("=== VAL / TEST transforms ===")
print(get_transforms('val'))

## Cell 8 — Augmentation sanity check

Display 5 training images side-by-side: **original** (left) vs **augmented** (right).  
Check that lesions are still visible and centred — no aggressive cropping, no colour inversions.

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from preprocessing import get_transforms, IMAGENET_MEAN, IMAGENET_STD

# Pick one sample per class from train_df for variety
samples = train_df.groupby('dx').first().reset_index()

train_tf = get_transforms('train')

def denorm(tensor):
    """Reverse ImageNet normalisation for display."""
    mean = np.array(IMAGENET_MEAN)
    std  = np.array(IMAGENET_STD)
    img  = tensor.numpy().transpose(1, 2, 0)
    img  = std * img + mean
    return np.clip(img, 0, 1)

n = len(samples)
fig, axes = plt.subplots(n, 2, figsize=(6, n * 2.5))
fig.suptitle("Original (left) vs Augmented (right)", fontsize=13, y=1.01)

for i, row in samples.iterrows():
    original = Image.open(row['filepath']).convert('RGB')
    augmented_tensor = train_tf(original)

    axes[i, 0].imshow(original.resize((224, 224)))
    axes[i, 0].set_title(f"{row['dx']} — original", fontsize=8)
    axes[i, 0].axis('off')

    axes[i, 1].imshow(denorm(augmented_tensor))
    axes[i, 1].set_title(f"{row['dx']} — augmented", fontsize=8)
    axes[i, 1].axis('off')

plt.tight_layout()

out_path = os.path.join(REPO_DIR, "outputs", "figures", "augmentation_samples.png")
os.makedirs(os.path.dirname(out_path), exist_ok=True)
plt.savefig(out_path, dpi=100, bbox_inches='tight')
plt.show()
print(f"Saved → {out_path}")

## Cell 9 — Build DataLoaders and verify batch shapes

In [ ]:
from preprocessing import make_dataloaders

loaders = make_dataloaders(train_df, val_df, test_df, batch_size=32, num_workers=2)

for split, loader in loaders.items():
    images, labels = next(iter(loader))
    print(f"{split:<6} — batches: {len(loader):>4} | batch shape: {tuple(images.shape)} | labels shape: {tuple(labels.shape)}")

## Stage 2 complete ✓

**Before moving to Stage 3, verify all boxes below:**

- [ ] `Train: ~7010  Val: ~1502  Test: ~1503` (approximately 70/15/15)
- [ ] Class % table shows all three splits are within ~1% of the Full dataset percentages
- [ ] Class weights printed — `df` or `vasc` should have the highest weight
- [ ] Augmented images look like the originals — lesions visible, colours natural, not cropped
- [ ] Batch shape is `(32, 3, 224, 224)` for images and `(32,)` for labels

**Next: Stage 3 — Baseline CNN training with `class_weights` passed to `CrossEntropyLoss`.**

---

### What was built and why

| Decision | Choice | Reason |
|---|---|---|
| Split ratio | 70/15/15 | Rarest class (df=115) needs ~17 images per eval set for stable recall |
| Stratification | on `dx` | Guarantees every class is proportionally represented in all splits |
| Augmentation | flip + rotate + small color jitter | Simulates real camera variation; preserves diagnostic features |
| No aggressive crop | excluded | Could cut off lesion boundary (clinically meaningful shape) |
| ImageNet normalisation | used for all splits | Required for pretrained backbones in Stage 4; no cost to use now |
| Class weights | `N / (C × n_c)` | Inverse frequency — forces loss to treat rare classes equally |
| Weights from train only | yes | Avoids data leakage — val/test distributions must stay unseen |

---